In [1]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from imblearn.under_sampling import RandomUnderSampler
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.metrics import classification_report
from torch.autograd import detect_anomaly
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, \
    GenerationConfig
import google.generativeai as genai
from google.generativeai import types
import os
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import google.api_core.exceptions
from dotenv import load_dotenv
from openai import OpenAI
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from accelerate import Accelerator

/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
detect_train_df = pd.read_csv('../data/detect_train.csv')
detect_train_dataset = Dataset.from_pandas(detect_train_df)

detect_test_df = pd.read_csv('../data/detect_test.csv')
detect_test_dataset = Dataset.from_pandas(detect_test_df)

# detect_train_balanced_df = pd.read_csv('../data/detect_train_balanced.csv')
# detect_train_balanced_dataset = Dataset.from_pandas(detect_train_balanced_df)

detect_n_shot_df = pd.read_csv('../data/detect_n_shot.csv')
detect_n_shot_dataset = Dataset.from_pandas(detect_n_shot_df)

# Classification Dataset

In [4]:
classify_train_df = pd.read_csv('../data/classify_train.csv')
classify_train_dataset = Dataset.from_pandas(classify_train_df)

classify_test_df = pd.read_csv('../data/classify_test.csv')
classify_test_dataset = Dataset.from_pandas(classify_test_df)

classify_n_shot_df = pd.read_csv('../data/classify_n_shot.csv')
classify_n_shot_dataset = Dataset.from_pandas(classify_n_shot_df)

In [12]:
DETECTION_TEMPLATE = PromptTemplate(
    name="Manually Crafted",
    definition="You are Code Expert trained to detect Self-Admitted Technical Debt (SATD) in Java test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional information indicating the need for future improvement. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Classify by labelling it as 'yes' if the comment include a strong indication of Self-Admitted Technical Debt otherwise label it as 'no', do not return reason. Do not provide a reason for the classification.",
    n_shot_template="""
    <EXAMPLE>
    Comment: {{ text }}
    {% if cot -%}
    Reason: {{ cot }}
    {% endif -%}
    Label: {{ label }}
    </EXAMPLE>""",
    line_m_before=3,
    line_n_after=3
)
DEFAULT_DETECTION_CLASS = 'no'

In [14]:
# Detect with google/flan-t5-small N-Shots

In [49]:
flan_t5_small_detection_model = HuggingFaceModel('detect', 'google/flan-t5-small', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
flan_t5_small_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_t5_small_detection_model.predict(detect_test_dataset,  DETECTION_TEMPLATE,
                              TrainStrategy.N_SHOT_SIMILAR,
                              4, verbose=False))


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

detect with flan-t5-small


Token indices sequence length is longer than the specified maximum sequence length for this model (635 > 512). Running this sequence through the model will result in indexing errors


Test Result:
              precision    recall  f1-score   support

          no      0.977     0.996     0.986      7592
         yes      0.029     0.006     0.009       177

    accuracy                          0.973      7769
   macro avg      0.503     0.501     0.498      7769
weighted avg      0.956     0.973     0.964      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.987     0.996     0.991      7495
         yes      0.029     0.010     0.015       103

    accuracy                          0.982      7598
   macro avg      0.508     0.503     0.503      7598
weighted avg      0.974     0.982     0.978      7598



# Detect with google/flan-t5-base N-Shots

In [14]:
flan_t5_base_detection_model = HuggingFaceModel('detect', 'google/flan-t5-base', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
flan_t5_base_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_t5_base_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE,
                              TrainStrategy.N_SHOT_SIMILAR,
                              4, verbose=False))


detect with flan-t5-base


Token indices sequence length is longer than the specified maximum sequence length for this model (614 > 512). Running this sequence through the model will result in indexing errors


Test Result:
              precision    recall  f1-score   support

          no      0.975     0.445     0.612      7592
         yes      0.021     0.514     0.041       177

    accuracy                          0.447      7769
   macro avg      0.498     0.480     0.326      7769
weighted avg      0.953     0.447     0.599      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.979     0.442     0.609      7495
         yes      0.008     0.311     0.015       103

    accuracy                          0.440      7598
   macro avg      0.493     0.376     0.312      7598
weighted avg      0.966     0.440     0.601      7598



# Detect with google/flan-t5-large N-Shots

In [14]:
flan_t5_large_detection_model = HuggingFaceModel('detect', 'google/flan-t5-large', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
flan_t5_large_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_t5_large_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 4,
                              verbose=False))


detect with flan-t5-large


Token indices sequence length is longer than the specified maximum sequence length for this model (612 > 512). Running this sequence through the model will result in indexing errors


Test Result:
              precision    recall  f1-score   support

          no      0.998     0.843     0.914      7592
         yes      0.122     0.932     0.215       177

    accuracy                          0.845      7769
   macro avg      0.560     0.888     0.564      7769
weighted avg      0.978     0.845     0.898      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.998     0.844     0.915      7495
         yes      0.073     0.893     0.135       103

    accuracy                          0.844      7598
   macro avg      0.536     0.868     0.525      7598
weighted avg      0.986     0.844     0.904      7598



# Detect with google/flan-t5-xl N-Shots

In [14]:
flan_t5_xl_detection_model = HuggingFaceModel('detect', 'google/flan-t5-xl', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
flan_t5_xl_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_t5_xl_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 4,
                              verbose=False))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

detect with flan-t5-xl


Token indices sequence length is longer than the specified maximum sequence length for this model (612 > 512). Running this sequence through the model will result in indexing errors


Test Result:
              precision    recall  f1-score   support

          no      0.999     0.965     0.981      7592
         yes      0.383     0.938     0.544       177

    accuracy                          0.964      7769
   macro avg      0.691     0.951     0.763      7769
weighted avg      0.984     0.964     0.971      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.998     0.965     0.981      7495
         yes      0.258     0.893     0.401       103

    accuracy                          0.964      7598
   macro avg      0.628     0.929     0.691      7598
weighted avg      0.988     0.964     0.973      7598



# Detect with google/flan-t5-xxl N-Shots

In [ ]:
detect_flan_t5_xxl_model = HuggingFaceModel('detect', 'google/flan-t5-xxl', {'yes', 'no'}, DEFAULT_DETECTION_CLASS, DETECTION_TEMPLATE)
detect_flan_t5_xxl_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(detect_flan_t5_xxl_model.predict(detect_test_dataset, TrainStrategy.N_SHOT_TOP, 4,
                              verbose=False))


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

detect with flan-t5-xxl


Token indices sequence length is longer than the specified maximum sequence length for this model (612 > 512). Running this sequence through the model will result in indexing errors


# Detect with Gemini 2.0 Flash N-Shots

In [46]:
gemini_2_flash_detection_model = GeminiModel('detect', 'models/gemini-2.0-flash', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
gemini_2_flash_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(gemini_2_flash_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE, TrainStrategy.N_SHOT_TOP,
                           10, verbose=False))

detect with gemini-2.0-flash
Test Result:
              precision    recall  f1-score   support

          no      1.000     0.959     0.979      7592
         yes      0.363     0.994     0.532       177

    accuracy                          0.960      7769
   macro avg      0.681     0.977     0.755      7769
weighted avg      0.985     0.960     0.969      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      1.000     0.959     0.979      7495
         yes      0.249     0.990     0.398       103

    accuracy                          0.959      7598
   macro avg      0.624     0.975     0.688      7598
weighted avg      0.990     0.959     0.971      7598



# Detect with gpt-4o-mini N-Shots

In [102]:
chat_gpt4o_mini_detection_model = ChatGpt4Model('detect', 'gpt-4o-mini', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
chat_gpt4o_mini_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(chat_gpt4o_mini_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE,
                                      TrainStrategy.N_SHOT_TOP, 4, verbose=False))


detect with gpt-4o-mini
              precision    recall  f1-score   support

          no      1.000     0.600     0.750        20
         yes      0.000     0.000     0.000         0

    accuracy                          0.600        20
   macro avg      0.500     0.300     0.375        20
weighted avg      1.000     0.600     0.750        20

Merging Mismatch


'./cache/March 23, 2025, 04:07:07$detect_gpt-4o-mini.csv'

# Detect with gpt-4o N-Shots

In [14]:
chat_gpt4o_detection_model = ChatGpt4Model('detect', 'gpt-4o', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
chat_gpt4o_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(chat_gpt4o_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE,
                                 TrainStrategy.N_SHOT_TOP, 4, verbose=False))


detect with gpt-4o
Unknown Labels:
['classification?', 'analysis?', "'no'?", '```', '</example>', '```', '```', 'code.', 'request.', 'evaluate.', '```', 'debt.', 'classification.', '</example>', 'classification.', 'concerns.', 'classification.', 'you?', 'debt.', '</example>', '</example>', 'request.', 'no.', '```', '```', '</example>', 'itself.', '```', '```', '</example>', 'comment.', '```', 'comments.', 'classified?', 'evaluation.', 'accordingly.', 'request.', 'analysis.', 'request.', '```', '</example>', '```', '```', 'classification.', 'else.', 'analysis.', 'not.', 'confirmed.', "'no'.", 'evaluation.', '```', 'classified?', 'needed?', 'that.', '```no```', '```', 'classification.', 'request.', "'no'.", '</example>', 'analysis.', 'request.', 'evaluated.', 'them.', 'that.', '```no```', 'debt?', '```', '</example>', 'classification.', 'that.', 'that.', 'comments.', 'debt.', 'debt.', 'classification.', '```', 'request.', 'scenario.', '```', 'label.', '```', 'classification.', 'debt.', '

# Detect with `all-MiniLM-L6-v2` Embedding and Logistic Regression

In [42]:
sentence_embedded_model = SentenceEmbeddedLogisticsRegressionModel('detect', 'all-MiniLM-L6-v2', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
sentence_embedded_model.fit(detect_train_balanced_dataset)
print_classification_excluding_outlier_repository(sentence_embedded_model.predict(detect_test_dataset))

detect with all-MiniLM-L6-v2
              precision    recall  f1-score   support

          no      0.997     0.932     0.963      7592
         yes      0.229     0.864     0.363       177

    accuracy                          0.931      7769
   macro avg      0.613     0.898     0.663      7769
weighted avg      0.979     0.931     0.950      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.997     0.932     0.963      7495
         yes      0.134     0.767     0.228       103

    accuracy                          0.929      7598
   macro avg      0.565     0.849     0.595      7598
weighted avg      0.985     0.929     0.953      7598



# Detect with SATD Text Mining Based SATD Detector

Pretrained SATD detector

New SATD Detector

In [17]:
satd_detector = TextMiningBasedSatdDetectorModel('detect', 'new-satd-detector', {'yes', 'no'}, DEFAULT_DETECTION_CLASS, retrain=True)
satd_detector.fit(detect_train_dataset)
satd_detector.predict(detect_test_dataset)

Finish Saving classifier of train
detect with new-satd-detector
../cache/September 10, 2025, 13:32:05$detect_new-satd-detector.csv
Test Result:
              precision    recall  f1-score   support

          no      0.993     0.976     0.985      6855
         yes      0.293     0.598     0.393       112

    accuracy                          0.970      6967
   macro avg      0.643     0.787     0.689      6967
weighted avg      0.982     0.970     0.975      6967



'../cache/September 10, 2025, 13:32:05$detect_new-satd-detector.csv'

Baseline Model

Pattern

In [17]:
for mode in ['pretrained', 'new']:
    pattern_model = BaselineModel('detect', f'{mode}-Pattern', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    pattern_model.fit(detect_train_dataset)
    pattern_model.predict(detect_test_dataset)

detect with pretrained-Pattern
False
Running model Pattern in MTO
Preparing data for Pattern
Pattern prediction finished!
Method: Pattern
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
233, 5573, 78, 112431, 0.749, 0.040, 0.076, 0.934, 14.267
3, 109, 2, 6852, 0.600, 0.027, 0.051, 0.973, 36.318

../cache/September 09, 2025, 18:17:59$detect_pretrained-Pattern.csv
Test Result:
              precision    recall  f1-score   support

          no      0.984     1.000     0.992      6855
         yes      0.600     0.027     0.051       112

    accuracy                          0.984      6967
   macro avg      0.792     0.513     0.522      6967
weighted avg      0.978     0.984     0.977      6967

detect with new-Pattern
True
Running model Pattern in MTO
Preparing data for Pattern
Pattern prediction finished!
Method: Pattern
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
14, 390, 6, 25177, 0.700, 0.035, 0.066, 0.977, 43.334
3, 109, 2, 6852, 0.600, 0.027, 0.051, 0.973, 36.318

../cache/Sep

Text Mining(TM)

In [16]:
for mode in ['pretrained', 'new']:
    tm_model = BaselineModel('detect', f'{mode}-TM', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    tm_model.fit(detect_train_dataset)
    tm_model.predict(detect_test_dataset)

detect with pretrained-TM
False
Running model TM in MTO
Preparing data for Pattern
../cache/baseline/pretrained/input/tm/data--train.arff
../cache/baseline/pretrained/input/tm/data--test.arff
Target: train, ../cache/baseline/pretrained/input/tm/data--train.arff
Target: test, ../cache/baseline/pretrained/input/tm/data--test.arff
Method: TM
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
2185, 809, 3866, 108643, 0.361, 0.730, 0.483, 0.928, 12.930
45, 67, 328, 6526, 0.121, 0.402, 0.186, 0.867, 6.504

../cache/September 10, 2025, 11:29:52$detect_pretrained-TM.csv
Test Result:
              precision    recall  f1-score   support

          no      0.990     0.952     0.971      6855
         yes      0.121     0.402     0.186       112

    accuracy                          0.943      6967
   macro avg      0.555     0.677     0.578      6967
weighted avg      0.976     0.943     0.958      6967

detect with new-TM
True
Running model TM in MTO
Preparing data for Pattern
../cache/baseline/ne

NLP

In [17]:
for mode in ['pretrained', 'new']:
    nlp_model = BaselineModel('detect', f'{mode}-NLP', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    nlp_model.fit(detect_train_dataset)
    nlp_model.predict(detect_test_dataset)

detect with pretrained-NLP
True
Running model NLP in MTO


../cache/September 10, 2025, 11:36:48$detect_pretrained-NLP.csv
Test Result:

Method: NLP
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
1780, 1214, 926, 111583, 0.658, 0.595, 0.625, 0.961, 24.377
38, 74, 52, 6802, 0.422, 0.339, 0.376, 0.962, 25.261

              precision    recall  f1-score   support

          no      0.989     0.992     0.991      6855
         yes      0.422     0.339     0.376       112

    accuracy                          0.982      6967
   macro avg      0.706     0.666     0.684      6967
weighted avg      0.980     0.982     0.981      6967

detect with new-NLP
True
Running model NLP in MTO


Method: NLP
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
220, 184, 21, 25162, 0.913, 0.545, 0.682, 0.983, 56.815
64, 48, 35, 6819, 0.646, 0.571, 0.607, 0.975, 39.208

../cache/September 10, 2025, 11:37:05$detect_new-NLP.csv
Test Result:
              precision    recall  f1-score   support

          no      0.993     0.995     0.994      6855
         yes      0.646     0.571     0.607       112

    accuracy                          0.988      6967
   macro avg      0.820     0.783     0.800      6967
weighted avg      0.987     0.988     0.988      6967



MAT

In [21]:
for mode in ['pretrained', 'new']:
    mat_model = BaselineModel('detect', f'{mode}-MAT', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    mat_model.fit(detect_train_dataset)
    mat_model.predict(detect_test_dataset)

detect with pretrained-MAT
True
Running model MAT in MTO
Preparing data for Pattern
../cache/baseline/pretrained/input/tm/data--train.arff
../cache/baseline/pretrained/input/tm/data--test.arff
MAT prediction finished!
Method: MAT
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
4452, 1354, 1030, 111479, 0.812, 0.767, 0.789, 0.940, 15.549
59, 53, 2, 6852, 0.967, 0.527, 0.682, 0.983, 59.157

../cache/September 09, 2025, 18:20:57$detect_pretrained-MAT.csv
Test Result:
              precision    recall  f1-score   support

          no      0.992     1.000     0.996      6855
         yes      0.967     0.527     0.682       112

    accuracy                          0.992      6967
   macro avg      0.980     0.763     0.839      6967
weighted avg      0.992     0.992     0.991      6967

detect with new-MAT
True
Running model MAT in MTO
Preparing data for Pattern
../cache/baseline/new/input/tm/data--train.arff
../cache/baseline/new/input/tm/data--test.arff
MAT prediction finished!
Method: 

# Detect with `text-embedding-004` Embedding and Logistic Regression

In [20]:
gemini_embedded_004_model = GeminiEmbeddedLogisticsRegressionModel('detect', 'models/text-embedding-004', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
gemini_embedded_004_model.fit(detect_train_balanced_dataset)
print_classification_excluding_outlier_repository(gemini_embedded_004_model.predict(Dataset.from_pandas(detect_test_df[:])))

detect with text-embedding-004
Test Result:
              precision    recall  f1-score   support

          no      0.997     0.952     0.974      7592
         yes      0.295     0.870     0.441       177

    accuracy                          0.950      7769
   macro avg      0.646     0.911     0.707      7769
weighted avg      0.981     0.950     0.962      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.997     0.953     0.975      7495
         yes      0.186     0.777     0.301       103

    accuracy                          0.951      7598
   macro avg      0.592     0.865     0.638      7598
weighted avg      0.986     0.951     0.965      7598



# Detect with `models/gemini-embedding-exp-03-07` Embedding and Logistic Regression

In [91]:
gemini_embedded_exp_model = GeminiEmbeddedLogisticsRegressionModel('detect', 'models/gemini-embedding-exp-03-07',
                                                                   {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
gemini_embedded_exp_model.fit(detect_train_balanced_dataset)
print_classification_excluding_outlier_repository(gemini_embedded_exp_model.predict(Dataset.from_pandas(detect_test_df[:])))

detect with gemini-embedding-exp-03-07
              precision    recall  f1-score   support

          no      0.996     0.988     0.992      1949
         yes      0.657     0.863     0.746        51

    accuracy                          0.985      2000
   macro avg      0.827     0.925     0.869      2000
weighted avg      0.988     0.985     0.986      2000

Merging Mismatch


'./cache/March 23, 2025, 03:38:26$detect_gemini-embedding-exp-03-07.csv'

# Classification Label Set

In [19]:
classification_label_set = set(classify_train_dataset['label']) | set(classify_test_dataset['label'])
classification_label_set

{'build',
 'code',
 'defect',
 'dependency',
 'design',
 'documentation',
 'how-to',
 'impractical-case',
 'multi',
 'refactor',
 'requirement',
 'skip-test',
 'subset-test',
 'superficial-test',
 'temporary-fix'}

# Classify with `all-MiniLM-L6-v2` Embedding and Logistic Regression

In [65]:
sentence_embedded_classification_model = SentenceEmbeddedLogisticsRegressionModel('classify', 'all-MiniLM-L6-v2', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
sentence_embedded_classification_model.fit(classify_train_dataset)
print_classification_excluding_outlier_repository(sentence_embedded_classification_model.predict(classify_test_dataset))

classify with all-MiniLM-L6-v2
./cache/March 27, 2025, 17:07:22$classify_all-MiniLM-L6-v2.csv
Test Result:
                  precision    recall  f1-score   support

           build      0.000     0.000     0.000         1
            code      0.000     0.000     0.000         7
          defect      1.000     0.050     0.095        20
      dependency      0.000     0.000     0.000         6
          design      0.000     0.000     0.000         2
   documentation      0.000     0.000     0.000         6
          how-to      0.286     0.091     0.138        22
impractical-case      0.000     0.000     0.000         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.672     0.978     0.796       182
       skip-test      0.000     0.000     0.000         7
superficial-test      0.000     0.000     0.000         8
   temporary-fix      0.250     0.136     0.176        22

        accuracy     

# Classify with `models/text-embedding-004` Embedding and Logistic Regression

In [68]:
gemini_embedded_004_classification_model = GeminiEmbeddedLogisticsRegressionModel('classify', 'models/text-embedding-004',  classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
gemini_embedded_004_classification_model.fit(classify_train_dataset)
print_classification_excluding_outlier_repository(gemini_embedded_004_classification_model.predict(classify_test_dataset))

classify with text-embedding-004
./cache/March 27, 2025, 17:10:18$classify_text-embedding-004.csv
Test Result:
                  precision    recall  f1-score   support

           build      0.000     0.000     0.000         1
            code      0.000     0.000     0.000         7
          defect      1.000     0.050     0.095        20
      dependency      0.000     0.000     0.000         6
          design      0.000     0.000     0.000         2
   documentation      0.000     0.000     0.000         6
          how-to      0.000     0.000     0.000        22
impractical-case      0.000     0.000     0.000         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.631     0.995     0.772       182
       skip-test      0.000     0.000     0.000         7
superficial-test      0.000     0.000     0.000         8
   temporary-fix      0.500     0.045     0.083        22

        accuracy 

# Classify with google/flan-t5-large n-Shots

In [18]:
classify_flan_t5_large_model = HuggingFaceModel('classify', 'google/flan-t5-large', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
classify_flan_t5_large_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(classify_flan_t5_large_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 0,
                              verbose=False))


Token indices sequence length is longer than the specified maximum sequence length for this model (688 > 512). Running this sequence through the model will result in indexing errors


classify with flan-t5-large
Unknown Labels:
['Deficit', 'Deficit', 'Deficit', 'Deficit']
Test Result:
                  precision    recall  f1-score   support

           build      1.000     1.000     1.000         1
            code      0.000     0.000     0.000         7
          defect      0.257     0.450     0.327        20
      dependency      0.143     0.167     0.154         6
          design      0.000     0.000     0.000         2
   documentation      1.000     0.333     0.500         6
          how-to      0.800     0.182     0.296        22
impractical-case      0.400     0.286     0.333         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.785     0.841     0.812       182
       skip-test      0.000     0.000     0.000         7
     subset-test      0.000     0.000     0.000         0
superficial-test      0.000     0.000     0.000         8
   temporary-fix      0.000

# Classify with google/flan-t5-xl n-Shots

In [15]:
classify_flan_t5_xl_model = HuggingFaceModel('classify', 'google/flan-t5-xl', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
classify_flan_t5_xl_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(classify_flan_t5_xl_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 0,
                              verbose=False))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (617 > 512). Running this sequence through the model will result in indexing errors


classify with flan-t5-xl
Test Result:
                  precision    recall  f1-score   support

           build      0.500     0.667     0.571         3
            code      1.000     0.111     0.200         9
          defect      0.750     0.300     0.429        20
      dependency      0.188     0.333     0.240         9
          design      0.000     0.000     0.000         0
   documentation      1.000     0.250     0.400         4
          how to      0.364     0.400     0.381        20
impractical case      0.000     0.000     0.000         1
           multi      0.043     0.333     0.077         3
        refactor      0.079     0.778     0.143         9
     requirement      0.955     0.626     0.756       171
       skip test      1.000     0.235     0.381        17
     subset test      1.000     0.500     0.667         2
superficial test      0.000     0.000     0.000         4
   temporary fix      1.000     0.100     0.182        20

        accuracy                

# Classify with google/flan-t5-xxl n-Shots

In [ ]:
classify_flan_t5_xxl_model = HuggingFaceModel('classify', 'google/flan-t5-xxl', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
classify_flan_t5_xxl_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(classify_flan_t5_xxl_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 0,
                              verbose=False))


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Token indices sequence length is longer than the specified maximum sequence length for this model (616 > 512). Running this sequence through the model will result in indexing errors


classify with flan-t5-xxl


# Classify with Gemini Flash 2.0 N-Shots

In [19]:
gemini_flash_2_classification_model = GeminiModel('classify', 'models/gemini-2.0-flash', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
gemini_flash_2_classification_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(gemini_flash_2_classification_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP,
                           0, verbose=False))

classify with gemini-2.0-flash
Test Result:
                  precision    recall  f1-score   support

           build      0.500     1.000     0.667         1
            code      0.150     0.429     0.222         7
          defect      0.667     0.600     0.632        20
      dependency      0.333     0.333     0.333         6
          design      0.000     0.000     0.000         2
   documentation      0.750     1.000     0.857         6
          how-to      0.700     0.636     0.667        22
impractical-case      1.000     0.286     0.444         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.926     0.896     0.911       182
       skip-test      0.417     0.714     0.526         7
     subset-test      0.000     0.000     0.000         0
superficial-test      0.400     0.500     0.444         8
   temporary-fix      0.500     0.182     0.267        22

        accuracy          

# Classify with gpt-4o-mini N-Shots

In [21]:
gpt_4o_mini_classification_model = ChatGpt4Model('classify', 'gpt-4o-mini', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
gpt_4o_mini_classification_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(gpt_4o_mini_classification_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP,
                           0, verbose=False))

classify with gpt-4o-mini
Unknown Labels:
['**dependency**', '**documentation**', '**how-to**', '**requirement**', '**temporary-fix**', '**how-to**', '**multi**', '**code**', '**documentation**', '**documentation**', '**multi**']
./cache/March 27, 2025, 18:01:56$classify_gpt-4o-mini.csv
Test Result:
                  precision    recall  f1-score   support

           build      0.200     1.000     0.333         1
            code      0.111     0.143     0.125         7
          defect      0.611     0.550     0.579        20
      dependency      0.188     0.500     0.273         6
          design      0.125     1.000     0.222         2
   documentation      0.333     0.833     0.476         6
          how-to      0.600     0.409     0.486        22
impractical-case      0.200     0.429     0.273         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.901     0.802     0.849       182
  

# Classify with gpt-4o N-Shots

In [22]:
gpt_4o_classification_model = ChatGpt4Model('classify', 'gpt-4o', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
gpt_4o_classification_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(gpt_4o_classification_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP,
                           0, verbose=False))

classify with gpt-4o
Unknown Labels:
['**temporary-fix**', '**temporary-fix**', '**dependency**', '**dependency**']
./cache/March 27, 2025, 18:07:16$classify_gpt-4o.csv
Test Result:
                  precision    recall  f1-score   support

           build      0.500     1.000     0.667         1
            code      0.300     0.429     0.353         7
          defect      0.636     0.350     0.452        20
      dependency      0.158     0.500     0.240         6
          design      0.167     0.500     0.250         2
   documentation      1.000     1.000     1.000         6
          how-to      0.722     0.591     0.650        22
impractical-case      0.214     0.429     0.286         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.905     0.890     0.898       182
       skip-test      0.600     0.857     0.706         7
     subset-test      0.000     0.000     0.000         0
super